# NYC Property Sales: Data Understanding

## Objective

Assess the structure, consistency, and quality of the 2024-2025 NYC property sales files before data cleaning and integration.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
DATA_DIR = Path("../data/raw")

files = sorted(DATA_DIR.glob("*/*.xlsx"))

print(f"Number of source files: {len(files)}")

for file in files:
    print(file)

Number of source files: 10
..\data\raw\2024\2024_bronx.xlsx
..\data\raw\2024\2024_brooklyn.xlsx
..\data\raw\2024\2024_manhattan.xlsx
..\data\raw\2024\2024_queens.xlsx
..\data\raw\2024\2024_staten_island.xlsx
..\data\raw\2025\2025_bronx.xlsx
..\data\raw\2025\2025_brooklyn.xlsx
..\data\raw\2025\2025_manhattan.xlsx
..\data\raw\2025\2025_queens.xlsx
..\data\raw\2025\2025_staten_island.xlsx


In [7]:
# inspect workbook structure
for file in files:
    workbook = pd.ExcelFile(file)
    print(f"{file.name}: {workbook.sheet_names}")

2024_bronx.xlsx: ['Bronx']
2024_brooklyn.xlsx: ['Brooklyn']
2024_manhattan.xlsx: ['Manhattan']
2024_queens.xlsx: ['Queens']
2024_staten_island.xlsx: ['Staten Island']
2025_bronx.xlsx: ['BRONX']
2025_brooklyn.xlsx: ['BROOKLYN']
2025_manhattan.xlsx: ['MANHATTAN']
2025_queens.xlsx: ['QUEENS']
2025_staten_island.xlsx: ['STATENISLAND']


## Initial Structure

The source files are inspected before loading the complete dataset to identify workbook structure, header placement, schema consistency, and other formatting characteristics that may affect ingestion.

In [8]:
# preview each file
for file in files:
    print(f"\n{'=' * 80}")
    print(file.name)
    print("=" * 80)

    preview = pd.read_excel(file, nrows=5)
    display(preview)


2024_bronx.xlsx


,BRONX ANNUAL SALES FOR CALENDAR YEAR 2024
0,All Sales From January 2024 - December 2024. P...
1,"For sales prior to the Final, Neighborhood Nam..."
2,"Sales after the Final Roll, Neighborhood Name ..."
3,Building Class Category is based on Building C...
4,Note: Condominium and cooperative sales are on...



2024_brooklyn.xlsx


,BROOKLYN ANNUAL SALES FOR CALENDAR YEAR 2024
0,All Sales From January 2024 - December 2024. P...
1,"For sales prior to the Final, Neighborhood Nam..."
2,"Sales after the Final Roll, Neighborhood Name ..."
3,Building Class Category is based on Building C...
4,Note: Condominium and cooperative sales are on...



2024_manhattan.xlsx


...

,MANHATTAN ANNUAL SALES FOR CALENDAR YEAR 2025
0,All Sales From January 2025 - December 2025. P...
1,"For sales prior to the Final, Neighborhood Nam..."
2,"Sales after the Final Roll, Neighborhood Name ..."
3,Building Class Category is based on Building C...
4,Note: Condominium and cooperative sales are on...



2025_queens.xlsx


,QUEENS ANNUAL SALES FOR CALENDAR YEAR 2025
0,All Sales From January 2025 - December 2025. P...
1,"For sales prior to the Final, Neighborhood Nam..."
2,"Sales after the Final Roll, Neighborhood Name ..."
3,Building Class Category is based on Building C...
4,Note: Condominium and cooperative sales are on...



2025_staten_island.xlsx


,STATEN ISLAND ANNUAL SALES FOR CALENDAR YEAR 2025
0,All Sales From January 2025 - December 2025. P...
1,"For sales prior to the Final, Neighborhood Nam..."
2,"Sales after the Final Roll, Neighborhood Name ..."
3,Building Class Category is based on Building C...
4,Note: Condominium and cooperative sales are on...


## Header Identification

The workbooks contain introductory metadata above the transaction records. The hearder row must therefore be identified before the files can be loaded as structured tables.

In [9]:
# inspect the 15 rows without assuming a header
sample_file = files[0]

header_preview = pd.read_excel(
    sample_file,
    header=None,
    nrows=15
)

display(header_preview)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,BRONX ANNUAL SALES FOR CALENDAR YEAR 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,All Sales From January 2024 - December 2024. P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"For sales prior to the Final, Neighborhood Nam...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Sales after the Final Roll, Neighborhood Name ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Building Class Category is based on Building C...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Note: Condominium and cooperative sales are on...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL\nUNITS,COMMERCIAL\nUNITS,TOTAL \nUNITS,LAND \nSQUARE FEET,GROSS \nSQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS\nAT TIME OF SALE,SALE PRICE,SALE DATE
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,BATHGATE,01 ONE FAMILY DWELLINGS,1,2907,24,NaN,A1,4090 PARK AVENUE,NaN,10457,1,0,1,2500,1474,1901,1,A1,0,2024-03-28 00:00:00
9,2,BATHGATE,01 ONE FAMILY DWELLINGS,1,3030,69,NaN,A1,4447 PARK AVENUE,NaN,10457,1,0,1,1668,1497,1899,1,A1,615000,2024-12-12 00:00:00


In [10]:
# locate the header in every file
header_rows = {}

for file in files:
    preview = pd.read_excel(file, header=None, nrows=15)
    matches = preview.iloc[:, 0].astype(str).str.strip().eq("BOROUGH")
    header_rows[file.name] = (
        preview.index[matches].tolist()
    )

header_rows

{'2024_bronx.xlsx': [6],
 '2024_brooklyn.xlsx': [6],
 '2024_manhattan.xlsx': [6],
 '2024_queens.xlsx': [6],
 '2024_staten_island.xlsx': [6],
 '2025_bronx.xlsx': [6],
 '2025_brooklyn.xlsx': [6],
 '2025_manhattan.xlsx': [6],
 '2025_queens.xlsx': [6],
 '2025_staten_island.xlsx': [6]}

In [11]:
# check the row immediately after each header
for file in files:
    preview = pd.read_excel(file, header=None, nrows=10)

    header_row = header_rows[file.name][0]
    following_row = preview.iloc[header_row + 1]

    print(
        f"{file.name}: "
        f"header row = {header_row}, "
        f"next row blank = {following_row.isna().all()}"
    )

2024_bronx.xlsx: header row = 6, next row blank = True
2024_brooklyn.xlsx: header row = 6, next row blank = True
2024_manhattan.xlsx: header row = 6, next row blank = True
2024_queens.xlsx: header row = 6, next row blank = True
2024_staten_island.xlsx: header row = 6, next row blank = True
2025_bronx.xlsx: header row = 6, next row blank = True
2025_brooklyn.xlsx: header row = 6, next row blank = True
2025_manhattan.xlsx: header row = 6, next row blank = True
2025_queens.xlsx: header row = 6, next row blank = True
2025_staten_island.xlsx: header row = 6, next row blank = True


## Structural Finding

All ten workbooks use a consistent layout. The first six rows contain source metadata, row 6 contains the tabular header, and the following is blank. A common `header=6` ingestion rule can therefore be applied across the source files.

In [12]:
# Load the raw datasets
datasets = {
    file.stem: pd.read_excel(file, header=6)
    for file in files
}

print(f"Datasets loaded: {len(datasets)}")

Datasets loaded: 10


In [15]:
# Dataset dimensions
dimensions = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
        for name, df in datasets.items()
    ]
)

dimensions

,dataset,rows,columns
0,2024_bronx,6204,21
1,2024_brooklyn,21995,21
2,2024_manhattan,17380,21
3,2024_queens,25004,21
4,2024_staten_island,7665,21
5,2025_bronx,6819,21
6,2025_brooklyn,23473,21
7,2025_manhattan,19725,21
8,2025_queens,27259,21
9,2025_staten_island,7797,21


In [16]:
# Total observations
print(f"Total raw rows: {dimensions['rows'].sum():,}")
print(f"Columns per dataset: {sorted(dimensions['columns'].unique())}")

Total raw rows: 163,321
Columns per dataset: [np.int64(21)]


# Schema Assessment

Before integration, the source datasets are compared for consistency in column names and inferred data types. Schema differences could indicate changes in source formatting between boroughs or years.

In [17]:
# Compare column names
# First establish the Bronx file as a reference schema
reference_name = "2024_bronx"
reference_columns = datasets[reference_name].columns.tolist()

print(f"Reference dataset: {reference_name}")
print(f"Number of columns: {len(reference_columns)}")

reference_columns

Reference dataset: 2024_bronx
Number of columns: 21


['BOROUGH',
 'NEIGHBORHOOD',
 'BUILDING CLASS CATEGORY',
 'TAX CLASS AT PRESENT',
 'BLOCK',
 'LOT',
 'EASE-MENT',
 'BUILDING CLASS AT PRESENT',
 'ADDRESS',
 'APARTMENT NUMBER',
 'ZIP CODE',
 'RESIDENTIAL\nUNITS',
 'COMMERCIAL\nUNITS',
 'TOTAL \nUNITS',
 'LAND \nSQUARE FEET',
 'GROSS \nSQUARE FEET',
 'YEAR BUILT',
 'TAX CLASS AT TIME OF SALE',
 'BUILDING CLASS\nAT TIME OF SALE',
 'SALE PRICE',
 'SALE DATE']

In [18]:
# Test schema consistency
schema_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "same_columns": df.columns.tolist() == reference_columns,
            "column_count": len(df.columns),
        }
        for name, df in datasets.items()
    ]
)

schema_check

,dataset,same_columns,column_count
0,2024_bronx,True,21
1,2024_brooklyn,True,21
2,2024_manhattan,True,21
3,2024_queens,True,21
4,2024_staten_island,True,21
5,2025_bronx,True,21
6,2025_brooklyn,True,21
7,2025_manhattan,True,21
8,2025_queens,True,21
9,2025_staten_island,True,21


In [20]:
# Inspect inferred data types
dtype_comparison = pd.DataFrame(
    {
        name: df.dtypes.astype(str)
        for name, df in datasets.items()
    }
)

dtype_comparison

,2024_bronx,2024_brooklyn,2024_manhattan,2024_queens,2024_staten_island,2025_bronx,2025_brooklyn,2025_manhattan,2025_queens,2025_staten_island
BOROUGH,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
NEIGHBORHOOD,str,str,str,str,str,str,str,str,str,str
BUILDING CLASS CATEGORY,str,str,str,str,str,str,str,str,str,str
TAX CLASS AT PRESENT,str,str,str,str,str,str,str,str,str,str
BLOCK,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
LOT,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
EASE-MENT,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
BUILDING CLASS AT PRESENT,str,str,str,str,str,str,str,str,str,str
ADDRESS,str,str,str,str,str,str,str,str,str,str
APARTMENT NUMBER,str,str,str,str,str,str,str,str,str,str


## Schema Finding

The ten source datasets contain 163,321 raw rows and share an identical 21-column schema, including consistent column ordering and inferred data types. This structural consistency supports row-wise concatenation after cleaning.

Several fields require further assessment before integration, including identifier-like columns stored as numeric values and column names containing embedded whitespace or line breaks.

In [22]:
# Missing values by dataset
missing_counts = pd.DataFrame(
    {
        name: df.isna().sum()
        for name, df in datasets.items()
    }
)

missing_counts

,2024_bronx,2024_brooklyn,2024_manhattan,2024_queens,2024_staten_island,2025_bronx,2025_brooklyn,2025_manhattan,2025_queens,2025_staten_island
BOROUGH,1,1,1,1,1,1,1,1,1,1
NEIGHBORHOOD,1,1,1,1,1,1,1,1,1,1
BUILDING CLASS CATEGORY,1,1,1,1,1,1,1,1,1,1
TAX CLASS AT PRESENT,1,1,1,1,1,1,1,1,1,1
BLOCK,1,1,1,1,1,1,1,1,1,1
LOT,1,1,1,1,1,1,1,1,1,1
EASE-MENT,6204,21995,17380,25004,7665,6819,23473,19725,27259,7797
BUILDING CLASS AT PRESENT,1,1,1,1,1,1,1,1,1,1
ADDRESS,1,1,1,1,1,1,1,1,1,1
APARTMENT NUMBER,5671,16394,9438,21086,6966,6252,17524,10339,22939,7340


In [24]:
# Missing percentages
missing_percentages = pd.DataFrame(
    {
        name: df.isna().mean().mul(100).round(2)
        for name, df in datasets.items()
    }
)

missing_percentages

,2024_bronx,2024_brooklyn,2024_manhattan,2024_queens,2024_staten_island,2025_bronx,2025_brooklyn,2025_manhattan,2025_queens,2025_staten_island
BOROUGH,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
NEIGHBORHOOD,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
BUILDING CLASS CATEGORY,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
TAX CLASS AT PRESENT,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
BLOCK,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
LOT,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
EASE-MENT,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
BUILDING CLASS AT PRESENT,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
ADDRESS,0.02,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.01
APARTMENT NUMBER,91.41,74.54,54.30,84.33,90.88,91.68,74.66,52.42,84.15,94.14


In [25]:
# Test for the blank observations
# we saw after the headers
empty_rows = pd.DataFrame(
    [
        {
            "dataset": name,
            "empty_rows": df.isna().all(axis=1).sum(),
        }
        for name, df in datasets.items()
    ]
)

empty_rows

,dataset,empty_rows
0,2024_bronx,1
1,2024_brooklyn,1
2,2024_manhattan,1
3,2024_queens,1
4,2024_staten_island,1
5,2025_bronx,1
6,2025_brooklyn,1
7,2025_manhattan,1
8,2025_queens,1
9,2025_staten_island,1


## Duplicate Assessment

Duplicate records are identified at the rw-data stage before any transformations are applied. Exact duplicates may reflect source formatting, repeated records, or legitimate transactions that require further investigation before removal.

In [27]:
# Exact duplicate rows
duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "duplicate_rows": df.duplicated().sum(),
            "duplicate_percentage": round(df.duplicated().mean() * 100, 2),
        }
        for name, df in datasets.items()
    ]
)

duplicate_summary

,dataset,duplicate_rows,duplicate_percentage
0,2024_bronx,0,0.0
1,2024_brooklyn,0,0.0
2,2024_manhattan,0,0.0
3,2024_queens,0,0.0
4,2024_staten_island,0,0.0
5,2025_bronx,0,0.0
6,2025_brooklyn,0,0.0
7,2025_manhattan,0,0.0
8,2025_queens,0,0.0
9,2025_staten_island,0,0.0


## Value Quality Assessment

Selected fields central to the research question are assessed for implausible, zero, or missing values. These checks inform later cleaning decisions without modifying the raw data.

In [28]:
# Zero and non-positive values
quality_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "zero_sale_price": (df["SALE PRICE"] == 0).sum(),
            "zero_land_sqft": (df["LAND \nSQUARE FEET"] == 0).sum(),
            "zero_gross_sqft": (df["GROSS \nSQUARE FEET"] == 0).sum(),
            "zero_year_built": (df["YEAR BUILT"] == 0).sum(),
        }
        for name, df in datasets.items()
    ]
)

quality_summary

,dataset,zero_sale_price,zero_land_sqft,zero_gross_sqft,zero_year_built
0,2024_bronx,2007,0,356,0
1,2024_brooklyn,8323,3,575,0
2,2024_manhattan,3965,12,43,0
3,2024_queens,9118,2,527,0
4,2024_staten_island,2558,0,338,0
5,2025_bronx,2395,2,371,0
6,2025_brooklyn,9171,4,577,0
7,2025_manhattan,4466,8,106,0
8,2025_queens,10709,3,532,0
9,2025_staten_island,2867,0,362,0


In [29]:
# Sale-date ranges
date_ranges = pd.DataFrame(
    [
        {
            "dataset": name,
            "min_sale_date": df["SALE DATE"].min(),
            "max_sale_date": df["SALE DATE"].max(),
        }
        for name, df in datasets.items()
    ]
)

date_ranges

,dataset,min_sale_date,max_sale_date
0,2024_bronx,2024-01-02,2024-12-31
1,2024_brooklyn,2024-01-01,2024-12-31
2,2024_manhattan,2024-01-01,2024-12-31
3,2024_queens,2024-01-01,2024-12-31
4,2024_staten_island,2024-01-01,2024-12-31
5,2025_bronx,2025-01-01,2025-12-31
6,2025_brooklyn,2025-01-01,2025-12-31
7,2025_manhattan,2025-01-01,2025-12-31
8,2025_queens,2025-01-01,2025-12-31
9,2025_staten_island,2025-01-01,2025-12-31


### Data Quality Findings

Missingness is not distributed uniformly across the variables or boroughs. `EASE-MENT` is entirely missing, while apartment identifiers, unit counts, and square-footage fields contain substantial missingness. Manhattan has particularly limited land and gross square-footage coverage.

Each source file also contains one completely blank observation introduced by the workbook layout. No exact records were identified.

Zero sale prices occur frequently and should not be interpreted as conventional market prices without further consideration. These records will require an explicit analytical treatment during data cleaning.

In [30]:
# Check borough provenance
borough_values = pd.DataFrame(
    [
        {
            "dataset": name,
            "borough_values": sorted(df["BOROUGH"].dropna().unique().tolist()),
        }
        for name, df in datasets.items()
    ]
)

borough_values

,dataset,borough_values
0,2024_bronx,[2.0]
1,2024_brooklyn,[3.0]
2,2024_manhattan,[1.0]
3,2024_queens,[4.0]
4,2024_staten_island,[5.0]
5,2025_bronx,[2.0]
6,2025_brooklyn,[3.0]
7,2025_manhattan,[1.0]
8,2025_queens,[4.0]
9,2025_staten_island,[5.0]


In [32]:
# Building categories
# assess categorical breadth
category_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "neighborhoods": df["NEIGHBORHOOD"].nunique(),
            "building_categories": df["BUILDING CLASS CATEGORY"].nunique(),
            "tax_classes": df["TAX CLASS AT PRESENT"].nunique(),
        }
        for name, df in datasets.items()
    ]
)

category_summary

,dataset,neighborhoods,building_categories,tax_classes
0,2024_bronx,37,35,9
1,2024_brooklyn,60,42,9
2,2024_manhattan,39,40,8
3,2024_queens,60,36,10
4,2024_staten_island,58,31,9
5,2025_bronx,37,36,9
6,2025_brooklyn,60,43,9
7,2025_manhattan,39,42,8
8,2025_queens,61,39,10
9,2025_staten_island,58,31,9


In [36]:
# Numeric range assessment
numeric_columns = [
    "SALE PRICE",
    "LAND \nSQUARE FEET",
    "GROSS \nSQUARE FEET",
    "RESIDENTIAL\nUNITS",
    "COMMERCIAL\nUNITS",
    "TOTAL \nUNITS",
    "YEAR BUILT",
]

numeric_ranges = pd.DataFrame(
    [
        {
            "variable": column,
            "min": min(df[column].min() for df in datasets.values()),
            "max": max(df[column].max() for df in datasets.values()),
        }
        for column in numeric_columns
    ]
)

numeric_ranges

,variable,min,max
0,SALE PRICE,0.0,1.080000e+09
1,LAND \nSQUARE FEET,0.0,7.446955e+06
2,GROSS \nSQUARE FEET,0.0,2.161994e+06
3,RESIDENTIAL\nUNITS,0.0,1.276000e+03
4,COMMERCIAL\nUNITS,0.0,4.720000e+02
5,TOTAL \nUNITS,0.0,1.276000e+03
6,YEAR BUILT,190.0,2.025000e+03


In [37]:
# Negative values
negative_counts = pd.DataFrame(
    [
        {
            "datasets": name,
            **{
                column.replace("\n", " ").strip(): (df[column] < 0).sum()
                for column in numeric_columns
            },
        }
        for name, df in datasets.items()
    ]
)

negative_counts

,datasets,SALE PRICE,LAND SQUARE FEET,GROSS SQUARE FEET,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,YEAR BUILT
0,2024_bronx,0,0,0,0,0,0,0
1,2024_brooklyn,0,0,0,0,0,0,0
2,2024_manhattan,0,0,0,0,0,0,0
3,2024_queens,0,0,0,0,0,0,0
4,2024_staten_island,0,0,0,0,0,0,0
5,2025_bronx,0,0,0,0,0,0,0
6,2025_brooklyn,0,0,0,0,0,0,0
7,2025_manhattan,0,0,0,0,0,0,0
8,2025_queens,0,0,0,0,0,0,0
9,2025_staten_island,0,0,0,0,0,0,0


### Temporal Coverage

Sale dates are consistent with the stated calendar year of each source file. The 2024 files span January-December 2024 and the 2025 files span January-December 2025, with no observed cross-year transactions in the recorded date ranges.

## Data Understanding Summary

The ten source workbooks contain 163,321 raw observations across New York City's five boroughs for 2024 and 2025. All files share the same 21-column schema, column ordering, header position, and inferred data-type structure, making them suitable for a consistent cleaning process followed by row-wise concatenation.

The initial assessment identified several issues that require treatment before analysis:

- each workbook contributes one completely blank row;
- `EASE-MENT` is entirely missing across the source data;
- several column names contain embedded line breaks and inconsistent whitespace;
- apartment numbers, unit counts, and square-footage fields contain substantial missingness;
- square-footage availability varies considerably by borough, particularly in Manhattan;
- zero sale prices occur frequently and require an explicit analytical treatment;
- some property-size fields contain zero values;
- `YEAR BUILT` contains at least one implausibly low value and requires validation;
- several identifier-like variables are represented as numeric values and should be reviewed before integration.

No exact duplicate rows or negative values were identified in the assessed fields. Sale dates are consistent with the stated year of each source file, and borough codes are internally consistent with the corresponding source files.

These findings define the cleaning and validation requirements for the next stage of the project.